In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import copy

from home_energy.load_esb import load_hdf
from home_energy.tariff import bill_summary, price_per_kwh

df = load_hdf("../data/raw/hdf_kwh_30min.csv")

with open("../config/system.yaml") as f:
    cfg = yaml.safe_load(f)

plans = cfg["tariffs"]
charges = cfg["regulated_charges"]
plan = plans["electric_ireland_current"]

In [ ]:
w = df.loc["2026-09-04":"2026-10-03"]
pd.Series(bill_summary(plan, w, days=30, charges=charges, start_date="2026-09-04", months=1))

In [ ]:
year = df.loc["2025-11-01":"2026-10-31"]
rows = {}
for month, chunk in year.groupby(year.index.to_period("M")):
   days = chunk.index.normalize().nunique()
   rows[str(month)] = bill_summary(plan, chunk, charges=charges, start_date="2025-11-01", days=days, months=1, apply_discounts=False)["net_cost"]
monthly = pd.Series(rows)
monthly.plot(kind="bar", title="Net cost per month, no discount (euros)")
monthly.sum()

In [ ]:
bill_summary(plan, year,charges=charges, start_date="2025-11-01", days=365, apply_discounts=False)["net_cost"]

In [ ]:
bill_summary(plan, year, charges=charges, start_date="2025-11-01", days=365, apply_discounts=True)["net_cost"]

In [ ]:
def vary(plan, change):
    """Return a copy of the plan after applying change(plan_copy)."""
    p = copy.deepcopy(plan)
    change(p)
    return p

def annual_cost(plan, usage=year, **kwargs):
    return bill_summary(plan, usage, days=365, charges=charges, **kwargs)["net_cost"]

In [ ]:
year = df.loc["2025-11-01":"2026-10-31"]
rates = [0, 0.05, 0.10, 0.15, 0.195, 0.25]
costs = []
for r in rates:
    p = vary(plan, lambda p: p["export_rates"][0].update(eur_per_kwh=r))
    costs.append(annual_cost(p, year, apply_discounts=False))

pd.Series(costs, index=rates).plot(marker="o", title="Annual net cost vs export rate")

In [ ]:
rate = plan["export_rates"][0]["eur_per_kwh"]

print("Exported kWh:", round(year["export_kwh"].sum()))
print("Export credit (euros):", round(year["export_kwh"].sum() * rate, 2))

monthly_credit = year["export_kwh"].resample("MS").sum() * rate
monthly_credit.plot(kind="bar", title="Export credit per month (euros)")

In [ ]:
def scale_import(label):
    def change(p, m):
        for rule in p["import_rates"]:
            if rule["label"] == label:
                rule["eur_per_kwh"] *= m
    return change

def scale_export(p, m):
    p["export_rates"][0]["eur_per_kwh"] *= m

def scale_standing(p, m):
    p["standing_charge_eur_per_day"] *= m

factors = {
    "night rate": scale_import("night"),
    "day rate": scale_import("day"),
    "peak rate": scale_import("peak"),
    "standing charge": scale_standing,
    "export rate": scale_export,
}

multipliers = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]
base = annual_cost(plan)

results = {}
for name, change in factors.items():
    costs = []
    for m in multipliers:
        p = copy.deepcopy(plan)
        change(p, m)
        costs.append(annual_cost(p) - base)
    results[name] = costs

sens = pd.DataFrame(results, index=[round((m - 1) * 100) for m in multipliers])

In [ ]:
ax = sens.plot(marker="o", figsize=(9, 5))
ax.set_xlabel("Change in input (%)")
ax.set_ylabel("Change in annual net cost (euros)")
ax.set_title("Sensitivity of the yearly bill (no discount)")
ax.axhline(0, color="grey", linewidth=0.8)
plt.show()

In [ ]:
effect = sens.loc[10].sort_values()
effect.plot(kind="barh", title="Effect on annual cost of +10% in each input (euros)")
plt.show()

In [ ]:
year["import_kwh"].groupby(price_per_kwh(year.index, plan)).sum()

In [ ]:
kwh = year["import_kwh"].groupby(price_per_kwh(year.index, plan)).sum()
pd.DataFrame({
    "kWh": kwh,
    "euros (before VAT)": kwh * kwh.index,
    "share of import (%)": kwh / kwh.sum() * 100,
}).round(1)

In [ ]:
night_rates = [0.12, 0.15, 0.1954, 0.22, 0.25]
export_rates = [0, 0.05, 0.10, 0.15, 0.195, 0.25]

grid = pd.DataFrame(index=night_rates, columns=export_rates, dtype=float)
for n in night_rates:
    for e in export_rates:
        p = copy.deepcopy(plan)
        for rule in p["import_rates"]:
            if rule["label"] == "night":
                rule["eur_per_kwh"] = n
        p["export_rates"][0]["eur_per_kwh"] = e
        grid.loc[n, e] = annual_cost(p)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(grid.values, aspect="auto", cmap="viridis_r")
ax.set_xticks(range(len(export_rates)), [f"{e:.3f}" for e in export_rates])
ax.set_yticks(range(len(night_rates)), [f"{n:.4f}" for n in night_rates])
ax.set_xlabel("Export rate (euros/kWh)")
ax.set_ylabel("Night rate (euros/kWh)")
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        ax.text(j, i, f"{grid.iloc[i, j]:.0f}", ha="center", va="center", color="white")
fig.colorbar(im, label="Annual net cost (euros)")
ax.set_title("Annual cost: night rate vs export rate (no discount)")
plt.show()

In [ ]:
import_cost = (year["import_kwh"] * price_per_kwh(year.index, plan)).sum()
flat_equivalent = import_cost / year["import_kwh"].sum()
flat_equivalent

In [ ]:
factors = [1.0, 0.9, 0.85, 0.8, 0.7]
costs = []
for f in factors:
    u = year.copy()
    u["export_kwh"] = u["export_kwh"] * f
    costs.append(annual_cost(plan, u))

pd.Series(costs, index=[f"{f:.0%}" for f in factors]).plot(
    marker="o", title="Annual cost if export is a share of this year's")
plt.ylabel("euros")
plt.show()

In [ ]:
cost = year["import_kwh"] * price_per_kwh(year.index, plan)
monthly_price = cost.resample("MS").sum() / year["import_kwh"].resample("MS").sum()
monthly_price.plot(kind="bar", title="Average price paid per kWh imported (euros)")
plt.show()

In [ ]:
price = price_per_kwh(year.index, plan)
monthly = pd.DataFrame({
    "day+peak import": year["import_kwh"].where(price > 0.25, 0).resample("MS").sum(),
    "night import": year["import_kwh"].where(price < 0.25, 0).resample("MS").sum(),
    "export": year["export_kwh"].resample("MS").sum(),
})
monthly.plot(kind="bar", figsize=(10, 4), title="Monthly kWh: dear import, night import and export")